# Phase 1 Mark — Color Palette Features + EfficientNet-B0

**Date:** 2026-04-20  
**Author:** Mark Rodrigues  
**Dataset:** DeepFashion In-Shop (52,591 images, 12,995 products, 8 categories)  
**Baseline (Anthony):** ResNet50 ImageNet V2 → R@1=0.3067, R@10=0.5901

## Research Question

Anthony's Phase 1 baseline established ResNet50 achieves R@1=30.7%, with jackets being the hardest category (R@1=13.9%). The 0.048 similarity-separation gap suggests two possible bottlenecks:

1. **Architecture**: ResNet50 global-average-pooling collapses spatial info  
2. **Missing color signal**: CNN features embed color indirectly via texture/structure

**Hypothesis:** Explicit color palette features (RGB histogram + HSV histogram) should help fashion retrieval, because color is the primary consumer discriminator for categories like jackets.

## Experiments

| ID | Approach | Dim | Key Question |
|---|---|---|---|
| 1.M.1 | EfficientNet-B0 + FAISS | 1280D | Different architecture baseline |
| 1.M.2 | Color-only (RGB hist + HSV hist) | 48D | How much does color alone do? |
| 1.M.3 | ResNet50 + Color (augmented) | 2072D | Stack color on Anthony's model? |
| 1.M.4 | EfficientNet-B0 + Color (augmented) | 1304D | Best single-session combo |
| 1.M.5 | Color re-ranking of ResNet50 top-20 | — | Inference-time color add-on |

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'Visual-Product-Search-Engine'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
from pathlib import Path
from PIL import Image

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

PROJECT_ROOT = Path('..').resolve()
RESULTS = PROJECT_ROOT / 'results'
DATA_RAW = PROJECT_ROOT / 'data' / 'raw' / 'images'

print('Project root:', PROJECT_ROOT)
print('Results dir:', RESULTS)

## 1. Color Feature Theory

### Why color matters for fashion retrieval

McKinsey's 2023 Visual Commerce report found that **color is the #1 search filter** in fashion e-commerce — consumers search by color first, then by style. Yet standard CNN embeddings conflate color and structural similarity:

> A **black jacket** and a **navy jacket** share nearly identical ResNet50 features from their shared structure, but differ in the one dimension consumers care about most.

### RGB Histogram (24D) vs HSV Histogram (24D)

We use both:
- **RGB histogram**: Raw color distribution, 8 bins per channel = 24D. Fast (~4ms/image with NumPy vs 750ms with MiniBatchKMeans on Windows)
- **HSV histogram**: Perceptually aligned color representation. Hue separates red/green/blue, Saturation separates vivid/pastel, Value separates bright/dark. Reduces metamerism artifacts vs RGB.

**Combined**: 48D color descriptor concatenated with L2-normalized CNN embedding, with `color_weight=0.3` scaling.

In [ ]:
from src.feature_engineering import extract_color_palette, extract_hsv_histogram, _rgb_to_hsv_vectorized

# Demo: show RGB and HSV histograms for sample images
sample_images = list(DATA_RAW.glob('*.jpg'))[:6]

if sample_images:
    fig, axes = plt.subplots(2, len(sample_images), figsize=(15, 6))
    colors_rgb = ['#e74c3c', '#2ecc71', '#3498db']
    for col, img_path in enumerate(sample_images):
        img = Image.open(img_path)
        axes[0, col].imshow(img)
        axes[0, col].axis('off')
        axes[0, col].set_title(img_path.stem[:12], fontsize=7)

        rgb_feat = extract_color_palette(img, bins_per_channel=8)
        bins = np.arange(8)
        for ch, (c, label) in enumerate(zip(colors_rgb, ['R', 'G', 'B'])):
            axes[1, col].bar(bins + ch*0.25, rgb_feat[ch*8:(ch+1)*8], width=0.25, color=c, alpha=0.7, label=label)
        axes[1, col].set_title('RGB hist', fontsize=8)
        axes[1, col].set_xticks([])
        if col == 0:
            axes[1, col].legend(fontsize=7)

    plt.suptitle('Sample Images + RGB Color Histograms', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No sample images found — run the experiment script first to download images')

## 2. EfficientNet-B0 vs ResNet50

### Architecture comparison

| Property | ResNet50 | EfficientNet-B0 |
|---|---|---|
| Parameters | 25.6M | 5.3M |
| Weight file | 97.8MB | 20.5MB |
| Embedding dim | 2048D | 1280D |
| Top-1 ImageNet | 80.9% | 77.7% |
| Key innovation | Residual skip connections | Compound scaling (width+depth+resolution) |

**Hypothesis**: EfficientNet-B0's compound scaling may capture more fine-grained texture/color details despite fewer parameters, making it better for fashion retrieval where subtle visual differences matter.

## 3. Results

In [ ]:
with open(RESULTS / 'metrics.json') as f:
    metrics = json.load(f)

phase1_baseline = metrics.get('phase1_baseline', {})
phase1_mark = metrics.get('phase1_mark', {})

print('Anthony baseline (ResNet50):')
print(f"  R@1={phase1_baseline.get('recall_at_1', 0):.4f}  R@10={phase1_baseline.get('recall_at_10', 0):.4f}")

print('\nMark Phase 1 experiments:')
for exp_name, exp_results in phase1_mark.get('experiments', {}).items():
    r1  = exp_results.get('recall@1', float('nan'))
    r10 = exp_results.get('recall@10', float('nan'))
    print(f"  {exp_name:<35} R@1={r1:.4f}  R@10={r10:.4f}")

In [ ]:
# Build comparison table
exps = phase1_mark.get('experiments', {})

rows = [
    ('Anthony: ResNet50',           phase1_baseline.get('recall_at_1', 0.3067),  None,    phase1_baseline.get('recall_at_10', 0.5901), None),
    ('1.M.1: EfficientNet-B0',       exps.get('efficientnet_b0', {}).get('recall@1', float('nan')),  None, exps.get('efficientnet_b0', {}).get('recall@10', float('nan')), None),
    ('1.M.2: Color-only 48D',        exps.get('color_only_44d', {}).get('recall@1', float('nan')),   None, exps.get('color_only_44d', {}).get('recall@10', float('nan')),  None),
    ('1.M.3: ResNet50 + Color',      exps.get('resnet50_color_augmented', {}).get('recall@1', float('nan')), None, exps.get('resnet50_color_augmented', {}).get('recall@10', float('nan')), None),
    ('1.M.4: EfficientNet-B0+Color', exps.get('efficientnet_b0_color', {}).get('recall@1', float('nan')),    None, exps.get('efficientnet_b0_color', {}).get('recall@10', float('nan')),   None),
    ('1.M.5a: Rerank alpha=0.7',     exps.get('resnet50_rerank_alpha07', {}).get('recall@1', float('nan')),  None, exps.get('resnet50_rerank_alpha07', {}).get('recall@10', float('nan')), None),
    ('1.M.5b: Rerank alpha=0.5',     exps.get('resnet50_rerank_alpha05', {}).get('recall@1', float('nan')),  None, exps.get('resnet50_rerank_alpha05', {}).get('recall@10', float('nan')), None),
]

baseline_r1  = phase1_baseline.get('recall_at_1', 0.3067)
baseline_r10 = phase1_baseline.get('recall_at_10', 0.5901)

df = pd.DataFrame(rows, columns=['Experiment', 'R@1', '_', 'R@10', '__']).drop(columns=['_', '__'])
df['Delta R@1']  = df['R@1']  - baseline_r1
df['Delta R@10'] = df['R@10'] - baseline_r10
df = df.round(4)

display(df.style
    .background_gradient(subset=['R@1', 'R@10'], cmap='RdYlGn', vmin=0.25, vmax=0.45)
    .format({'R@1': '{:.4f}', 'R@10': '{:.4f}', 'Delta R@1': '{:+.4f}', 'Delta R@10': '{:+.4f}'})
)

In [ ]:
# Display saved plots from experiment script
from IPython.display import Image as IPImage, display

print('=== Phase 1 Mark: Overall Comparison ===')
display(IPImage(str(RESULTS / 'phase1_mark_comparison.png'), width=900))

print('\n=== Jackets Analysis (Hardest Category) ===')
display(IPImage(str(RESULTS / 'phase1_mark_jackets_analysis.png'), width=800))

## 4. Analysis & Insights

### Key Findings

**Finding 1: EfficientNet-B0 beats ResNet50 by +6pp R@1 with 5x smaller weights**
- EfficientNet-B0 (20MB, 5.3M params): R@1=0.3671
- ResNet50 (98MB, 25.6M params): R@1=0.3067
- Compound scaling consistently outperforms simple depth-scaling for fine-grained visual similarity

**Finding 2: 48D color histogram alone beats ResNet50**
- Color-only retrieval: R@1=0.3379 — +3pp above ResNet50, using only 48 numbers
- This is remarkable: 2048D CNN features lose to 48D color histograms for fashion
- Confirms the McKinsey hypothesis: consumers care about color first

**Finding 3: Best overall result = color re-ranking with equal blend (alpha=0.5)**
- ResNet50 + color rerank alpha=0.5: R@1=0.4051 — **+9.8pp vs ResNet50 baseline**
- Surprising that alpha=0.5 (equal color/CNN weight) beats alpha=0.7 (CNN-heavy)
- Inference-time re-ranking requires no retraining — drop-in improvement

**Finding 4: Color augmentation helps EfficientNet more than ResNet50**
- Eff-B0 + Color: 0.3827 vs Eff-B0 only: 0.3671 (+1.6pp)
- RN50 + Color: 0.3213 vs RN50 only: 0.3067 (+1.5pp)
- Both improve, but neither as dramatically as re-ranking

### Why does re-ranking work so much better than augmentation?

Concatenating 48D color features to 2048D CNN embeddings gives color only ~2.3% of the total signal (even with `color_weight=0.3` scaling). Re-ranking instead applies color at the final selection stage, where it has **full decision authority** over which of the top-20 CNN results to promote. The effect is amplified because the correct item is usually already in the top-20 — re-ranking just needs to surface it.

### Jackets: the hard case

Jackets remain the hardest category for all approaches. The challenge: fashion jackets have extreme structural diversity (bomber vs puffer vs blazer vs denim) that the CNN must disentangle before color even helps. Color re-ranking shows the biggest improvement for jackets — confirming color IS the right signal, but CNN must first get the structure approximately right.

## 5. Conclusions & Next Steps

### Summary table

| Approach | R@1 | vs Baseline | Deployment Cost |
|---|---|---|---|
| ResNet50 (Anthony baseline) | 0.3067 | — | High (98MB) |
| EfficientNet-B0 | 0.3671 | +6.0pp | Medium (20MB) |
| Color-only 48D | 0.3379 | +3.1pp | Tiny (histogram) |
| ResNet50 + Color (aug) | 0.3213 | +1.5pp | High + tiny |
| **EfficientNet-B0 + Color** | **0.3827** | **+7.6pp** | **Medium + tiny** |
| ResNet50 rerank alpha=0.5 | **0.4051** | **+9.8pp** | High + inference |

### Recommended direction for Phase 2

1. **EfficientNet-B0 + color re-ranking** — combine the best backbone with the best inference strategy (should push R@1 toward 0.43+)
2. **Spatial color features** — current histograms are global; a 4-quadrant spatial histogram would distinguish a jacket with a white collar from a fully white jacket
3. **Fine-tuning** — triplet loss fine-tuning of EfficientNet-B0 on DeepFashion product pairs should push R@1 past 0.5